# Simple Examples

`BioacousticAnnotator` allows the user to configure almost every aspect of the app. Here we focus on the most minimal configurations for: [a player](#1.-Player-/-Visualizar), [data collection](#2.b-Data-Collection), and [model review](2.b-Model-Review). Each configuration below passes paramters directly to `BioacousticAnnotator`. Using a `config` file, rather than passing parameters directly will help produce cleaner, more reproducible workflows. [Section 3](#3.-Simple-Confgurations-Files) shows basic configuration files for the examples in [section 2](2.-Simple-Forms). See the [Bioacoustic Annotators Examples Notebook](./bioacoustic-annotator-Examples.ipynb) for a more advanced/realistic configurations.

```
Author: Brookie Guzder-Williams (bguzder-williams@berkeley.edu)
Affiliation: The Eric and Wendy Schmidt Center for Data Science & Environment
Website: https://dse.berkeley.edu/
License: BSD 3-Clause
```

For configuration options and advanced usage see the [documentation](https://github.com/SchmidtDSE/jupyter_bioacoustic/wiki) and the [geo-analysis example](geo-analysis-example.ipynb).

In [165]:
#| label: nb.simple-examples.setup
from jupyter_bioacoustic import BioacousticAnnotator, print_md

DATA = 'data/detections.N-dse.csv'
AUDIO = 'audio_url'
CATEGORIES = 'data/species_counts.N-dse.csv'

---

## 1. Player / Visualizar

Browse audio clips and view spectrograms — no form, no data collection. Useful for exploration and quality checks.

In [36]:
#| label: nb.simple-examples.1.player
#| placeholder: ../assets/nb/simple-examples.1.player.png
ba = BioacousticAnnotator(data=DATA, audio=AUDIO, display_columns=['common_name', 'confidence'])
ba.open()

<IPython.core.display.Javascript object>


---

## 2. Simple Forms

Here we show two simple forms created by passing a dictionary to the `form_config`:

- Data Collection: a simple form requiring the user to select a species for the clip
- Model Review: the user is first asked if the model prediction is valid.  if they select yes, the user may submit their response. If the user selects no, they must select what speicies is contained in the clip.

Note: `form_config` also accepts a path to yaml file. `form_config` may also be passed directly in a `config`-file (see [bioacoustic-annotator-examples](./bioacoustic-annotator-examples.ipynb)).


### 2.a Data Collection

Our only form element in this example is a `select` dropdown.  This example uses the following parameters:

- `label`: display label; used as column name if column is omitted
- `column`: column written to output file (defaults to label)
- `required`: if true, submit button is disabled until value is set
- `items`: the `items` elements can take several formats, lists of strings or dicts for example. in this case we specifiy a path to a CSV and the column in the CSV to popluate the list with.

Additionally, Now that we are collecting data we need to be able to associate it with the selected row in the clip table.  to do this we need to define a `data_index_column`, the id-column in the source dataset. and `output_index_column`, the associated id-column in the output dataset (this will default to `data_index_column`.


In [40]:
#| label: nb.simple-examples.2a.annotation
#| placeholder: ../assets/nb/simple-examples.2a.annotation.png
BioacousticAnnotator(
    data=DATA,
    audio=AUDIO,
    display_columns=['common_name', 'confidence'],
    data_index_column='id',
    output_index_column='detection_id',
    form_config={
        'select': {
            'label': 'species',
            'column': 'common_name',
            'required': True,
            'items': {
                'path': CATEGORIES,
                'value': 'common_name',
            },
        },
    },
).open()

<IPython.core.display.Javascript object>

### 2.b Model Review

The form in this example in includes two elements: a "is valid" select-box, and an additional form (containing a single select-box) If the user chooses that the model has incorrectly identified the species.

Note that the `correction_form` select-box is similar to above with some additional elements under `select.items`.  Namely:

- `filter_box`: `(bool)` if set to  true an additional textbox appears next to the dropdown that allows the user to filter through options. This can be extremely useful when there are 100s of speicies to choose from
- `not_available`: `(bool|str)` if truthy adds an additonal "not_available" value (in this case "Unknown Species") to the list
- used a `[[]]-template` to included the predicted `common_name` in the select label

In [47]:
#| label: nb:simple-examples.2b.review
#| placeholder: ../assets/nb/simple-examples.2b.review.png
BioacousticAnnotator(
    data=DATA,
    audio=AUDIO,
    display_columns=['common_name', 'confidence'],
    data_index_column='id',
    output_index_column='detection_id',
    form_config={
        'select': {
            'label': 'Is [[common_name]] Valid',
            'column': 'is_valid',
            'required': True,
            'items': [
                { 'value': 'yes'},
                { 'value': 'no', 'form': 'correction_form'},
            ],
        },
        'dynamic_forms': {
            'correction_form': [
                {'select': {
                    'label': 'corrected species',
                    'column': 'corrected_common_name',
                    'items': {
                        'path': CATEGORIES,
                        'value': 'common_name',
                        'required': True,
                        'filter_box': True,
                        'not_available': 'Unknown Species',
                    },
                }},
            ],
        }
    },
).open()

<IPython.core.display.Javascript object>

---

## 3. Simple Confgurations Files

Here we repeat the two examples above using configuration files.  

### 3.a `form_config` example

In [140]:
#| label: nb.simple-examples.3a.annotation
#| placeholder: ../assets/nb/simple-examples.3a.annotation.png
ba = BioacousticAnnotator(
    data=DATA,
    audio=AUDIO,
    display_columns=['common_name', 'confidence'],
    data_index_column='id',
    output_index_column='detection_id',
    form_config='annotator_config/forms/simple-examples-3.yaml'
)
ba.describe()

---

**Configuration Files**

- Form: annotator_config/forms/simple-examples-3.yaml


---

**Configuration**

audio: audio_url
data:
  index_column: id
  path: data/detections.N-dse.csv
display_columns:
- common_name
- confidence
form_config:
  dynamic_forms:
  - confirmed_form:
    - select:
        column: reviewer_confidence
        items:
        - low
        - medium
        - high
        label: confidence
    - textbox:
        column: notes
        label: notes
  - rejected_form:
    - select:
        column: corrected_common_name
        items:
          custom_value: true
          filter_box: true
          not_available:
            label: Unknown Species
            value: unknown
          path: data/species_counts.N-dse.csv
          value: common_name
        label: corrected species
        required: true
    - select:
        column: rejection_reason
        items:
        - noise
        - wrong species
        - overlapping signals
        - too faint
        - other
        label: rejection reason
        required: true
    - number:
        column: signal_quality
       

---

In [141]:
ba.open()

<IPython.core.display.Javascript object>

### 3.b `config` example

The advantage to `config` over `form_config` is that you can pass the full range of parameters to `BioacousticAnnotator`.  Note however, in this example we are passing only `form_config`.  You can dive deeper into the full set of parameters [here](./bioacoustic-annotator-examples.ipynb).  We have kept `data` and `audio` as inline params, since this would allow a user to use the same config for multiple different `data` and `audio` values.  

In [147]:
#| label: nb:simple-examples.3b.review
#| placeholder:  ../assets/nb/simple-examples.3b.review.png
ba = BioacousticAnnotator(
    data=DATA,
    audio=AUDIO,
    config='annotator_config/config/simple-examples-3.yaml'
)
ba.describe()

---

**Configuration Files**

- Config: annotator_config/config/simple-examples-3.yaml
- Form: annotator_config/forms/simple-examples-3.yaml


---

**Configuration**

audio: audio_url
capture: Save Spectrogram
capture_dir: spectrograms
data:
  index_column: id
  path: data/detections.N-dse.csv
display_columns:
- common_name
- confidence
- rank
- county
form_config:
  dynamic_forms:
  - confirmed_form:
    - select:
        column: reviewer_confidence
        items:
        - low
        - medium
        - high
        label: confidence
    - textbox:
        column: notes
        label: notes
  - rejected_form:
    - select:
        column: corrected_common_name
        items:
          custom_value: true
          filter_box: true
          not_available:
            label: Unknown Species
            value: unknown
          path: data/species_counts.N-dse.csv
          value: common_name
        label: corrected species
        required: true
    - select:
        column: rejection_reason
        items:
        - noise
        - wrong species
        - overlapping signals
        - too faint
        - other
        label: rejection reason
       

---

In [148]:
ba.open()

<IPython.core.display.Javascript object>

### 3.c `project` example

A `project` configuration is similar to `config` except it must include all required fields (ie `data.{path|uri|..., ident_column}`, `audio.{path|...}`) and you can pass a `config` path to be loaded. 

In [170]:
print_md((
    "Here we will try and load the same config file from the previous example directly as a project. "
    "Loading will fail because the project is not fully specified"
))
try:
    BioacousticAnnotator(project='annotator_config/config/simple-examples-3.yaml')
except Exception as e:
    print_md('---')
    print('ERROR:', e)
    print_md('---')

Here we will try and load the same config file from the previous example directly as a project. Loading will fail because the project is not fully specified

---

ERROR: data dict must have exactly one of {'sql', 'api', 'url', 'path', 'uri'}, got: none


---

In [171]:
#| label: nb:simple-examples.3b.review
#| placeholder:  ../assets/nb/simple-examples.3b.review.png
ba = BioacousticAnnotator('annotator_config/project/simple-examples-3.yaml')
ba.describe()

---

**Configuration Files**

- Project: annotator_config/project/simple-examples-3.yaml
- Config: annotator_config/config/simple-examples-3.yaml
- Form: annotator_config/forms/simple-examples-3.yaml


---

**Configuration**

audio:
  column: audio_url
capture: Save Spectrogram
capture_dir: spectrograms
data:
  index_column: id
  path: data/detections.N-dse.csv
display_columns:
- common_name
- confidence
- rank
- county
form_config:
  dynamic_forms:
  - confirmed_form:
    - select:
        column: reviewer_confidence
        items:
        - low
        - medium
        - high
        label: confidence
    - textbox:
        column: notes
        label: notes
  - rejected_form:
    - select:
        column: corrected_common_name
        items:
          custom_value: true
          filter_box: true
          not_available:
            label: Unknown Species
            value: unknown
          path: data/species_counts.N-dse.csv
          value: common_name
        label: corrected species
        required: true
    - select:
        column: rejection_reason
        items:
        - noise
        - wrong species
        - overlapping signals
        - too faint
        - other
        label: rejection reas

---

In [172]:
ba.open()

<IPython.core.display.Javascript object>